# 📊 Exploratory Data Analysis (EDA)

**Objective**: To explore, visualize, and understand the core datasets driving the Drug Interaction System: `TWOSIDES` (Side effects/ADRs), `DrugBank` (Known Interactions), and the final `Merged` dataset.

**Strategy**: We will analyze each dataset separately to understand their specific data types, missing values, and numeric distributions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os

# Set visualization styles
sns.set_theme(style="whitegrid", palette="pastel")
plt.rcParams["figure.figsize"] = (12, 6)

def get_sample_skip_func(filepath, target_sample_size=1000000):
    """Helper to safely sample giant CSVs without loading everything to RAM."""
    with open(filepath, 'r', encoding='utf-8') as f:
        total_rows = sum(1 for line in f)
    
    skip_prob = 1 - (target_sample_size / total_rows) if total_rows > target_sample_size else 0
    return lambda i: i > 0 and random.random() < skip_prob, total_rows

print("Libraries and helpers loaded successfully.")

## 1. DrugBank Dataset (`db_drug_interactions.csv`)
Contains purely categorical/text data (`Drug 1`, `Drug 2`, `Interaction Description`).

In [ ]:
db_path = "../data/converted/db_drug_interactions.csv"
df_db = pd.read_csv(db_path)

print(f"DrugBank Data Shape: {df_db.shape[0]:,} rows, {df_db.shape[1]} columns")
print("\n--- Columns & Data Types ---")
print(df_db.dtypes)

display(df_db.head())

In [ ]:
# Visualize Interaction Text Lengths
if 'Interaction Description' in df_db.columns:
    df_db['desc_length'] = df_db['Interaction Description'].astype(str).str.len()
    
    plt.figure(figsize=(10, 5))
    sns.histplot(df_db['desc_length'], bins=50, kde=True, color="skyblue")
    plt.title('DrugBank: Distribution of Interaction Description Lengths')
    plt.xlabel('Number of Characters')
    plt.ylabel('Frequency')
    plt.show()
    
    print("\n--- Summary Stats of Text Length ---")
    print(df_db['desc_length'].describe())

## 2. TWOSIDES Dataset (`TWOSIDES-002.csv`)
A massive dataset detailing side effects (`condition_concept_name`) and statistical reporting metrics (`PRR`, `PRR_error`, `mean_reporting_frequency`). We load a sample to inspect.

In [ ]:
twosides_path = "../data/converted/TWOSIDES-002.csv"
skip_func, total_ts_rows = get_sample_skip_func(twosides_path, target_sample_size=1000000)

print(f"Total TWOSIDES rows: {total_ts_rows:,}. Loading ~1M sample...")
df_twosides = pd.read_csv(twosides_path, skiprows=skip_func, low_memory=False)

# Convert metrics to numeric (they are often loaded as objects due to dirty headers/values)
numeric_cols = ['PRR', 'PRR_error', 'mean_reporting_frequency', 'A', 'B', 'C', 'D']
for col in numeric_cols:
    if col in df_twosides.columns:
        df_twosides[col] = pd.to_numeric(df_twosides[col], errors='coerce')

print(f"\nSampled TWOSIDES Data Shape: {df_twosides.shape[0]:,} rows, {df_twosides.shape[1]} columns")
print("\n--- Numerical Columns Summary ---")
display(df_twosides[numeric_cols].describe())

display(df_twosides.head())

In [ ]:
# Top 15 Condition Concept Names (Side Effects)
if 'condition_concept_name' in df_twosides.columns:
    top_conditions = df_twosides['condition_concept_name'].value_counts().head(15)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=top_conditions.values, y=top_conditions.index, palette="magma")
    plt.title('TWOSIDES: Top 15 Most Frequent Side Effects (Sample)')
    plt.xlabel('Frequency in Sample')
    plt.show()

# Distribution of PRR (Proportional Reporting Ratio)
if 'PRR' in df_twosides.columns:
    plt.figure(figsize=(10, 5))
    # Limiting to PRR < 50 for visualization as it has extreme outliers
    sns.histplot(df_twosides[df_twosides['PRR'] < 50]['PRR'], bins=50, kde=True, color="coral")
    plt.title('TWOSIDES: Distribution of Proportional Reporting Ratio (PRR < 50)')
    plt.xlabel('PRR Score')
    plt.ylabel('Frequency')
    plt.show()

## 3. Merged Dataset (`merged_drug_interactions-001.csv`)
This aggregates both text descriptions from DrugBank and numeric side-effect metrics from TWOSIDES.

In [ ]:
merged_path = "../data/converted/merged_drug_interactions-001.csv"
skip_func, total_merged_rows = get_sample_skip_func(merged_path, target_sample_size=1000000)

print(f"Total Merged rows: {total_merged_rows:,}. Loading ~1M sample...")
df_merged = pd.read_csv(merged_path, skiprows=skip_func, low_memory=False)

print(f"\nSampled Merged Data Shape: {df_merged.shape[0]:,} rows, {df_merged.shape[1]} columns")
print("\n--- Missing Values in Merged Data ---")
print(df_merged.isnull().sum())

display(df_merged.head())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# 1. Top Interacting Drugs in Merged Data
all_merged_drugs = pd.concat([df_merged['Drug 1'], df_merged['Drug 2']])
top_merged_drugs = all_merged_drugs.value_counts().head(15)

sns.barplot(x=top_merged_drugs.values, y=top_merged_drugs.index, palette="viridis", ax=ax[0])
ax[0].set_title('Merged: Top 15 Most Frequent Drugs')
ax[0].set_xlabel('Number of Mentions in Sample')

# 2. Correlation of Numeric Metrics (A, B, C, D, PRR)
numeric_merged = df_merged[['A', 'B', 'C', 'D', 'PRR', 'mean_reporting_frequency']].corr()
sns.heatmap(numeric_merged, annot=True, cmap="coolwarm", fmt=".2f", ax=ax[1])
ax[1].set_title('Merged: Correlation of Reporting Metrics')

plt.tight_layout()
plt.show()

## 4. Summary & Verification Notes
- **DrugBank**: Focuses purely on semantic information (`Interaction Description`). Ideal for NLP-based feature extraction (Task 1.2).
- **TWOSIDES**: Focuses on `condition_concept_name` and computational validity via `PRR` and `mean_reporting_frequency`.
- **Merged**: Successfully brings unstructured text and structured numerical validity (PRR) into a massive 5.6GB space. Ready for Model feature engineering.